# function plotting, cog analysis

uses `master_groups.csv` and `master_strains.csv`. toggle `SHOW_FILTERED` to
switch between filtered and unfiltered data.

figures produced:

- figure 1: `PV_per_Mb_boxplot.png`
- figure 2: `COG_broad_role_pct_and_cps.png`
- figure 3a: `COG_3A_pct_heatmap.png`
- figure 3b: `COG_3B_cps_heatmap.png`
- figure 3c: `COG_3C_letter_key.png`


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.colors as mcolors
import sys
from pathlib import Path
from IPython.display import display, HTML
from matplotlib import rcParams


plt.rcParams["font.family"]     = "sans-serif"
plt.rcParams["font.sans-serif"] = ["Arial"]

cwd = Path.cwd().resolve()
PROJECT_ROOT = next((p for p in [cwd, *cwd.parents] if (p / "Standards" / "colors.py").exists()), cwd)

sys.path.append(str(PROJECT_ROOT / "Standards"))
from colors import color_map, genus_order

control_color_map = {
    "Campylobacter": "#2d2d2d",
    "Escherichia":   "#555555",
    "Brucella":      "#888888",
}

EXCLUDE_COGS = ["Unknown", "Function unknown", "General function prediction only"]

COG_DESCRIPTIONS = {
    "J": "Translation",
    "A": "RNA processing and modification",
    "K": "Transcription",
    "L": "Replication, recombination and repair",
    "B": "Chromatin structure and dynamics",
    "D": "Cell cycle control and division",
    "Y": "Nuclear structure",
    "V": "Defense mechanisms",
    "T": "Signal transduction",
    "M": "Cell wall/membrane/envelope biogenesis",
    "N": "Cell motility",
    "Z": "Cytoskeleton",
    "W": "Extracellular structures",
    "U": "Intracellular trafficking and secretion",
    "O": "Post-translational modification and chaperones",
    "X": "Mobilome: prophages and transposons",
    "C": "Energy production and conversion",
    "G": "Carbohydrate transport and metabolism",
    "E": "Amino acid transport and metabolism",
    "F": "Nucleotide transport and metabolism",
    "H": "Coenzyme transport and metabolism",
    "I": "Lipid transport and metabolism",
    "P": "Inorganic ion transport and metabolism",
    "Q": "Secondary metabolite biosynthesis and catabolism",
    "R": "General function prediction only",
    "S": "Function unknown",
}

BROAD_ROLE_MAP = {
    # Surface & Defense — canonical PV substrates
    "M": "Surface & Defense",
    "N": "Surface & Defense",
    "W": "Surface & Defense",
    "V": "Surface & Defense",
    # Environmental Response — regulatory switching
    "T": "Env. Response",
    "K": "Env. Response",
    # Adaptive Metabolism — contingent, environmentally-driven
    "P": "Adaptive Metabolism",
    "G": "Adaptive Metabolism",
    "Q": "Adaptive Metabolism",
    "E": "Adaptive Metabolism",
    "C": "Adaptive Metabolism",
    "H": "Adaptive Metabolism",
    "I": "Adaptive Metabolism",
    # Core Cellular Machinery 
    "F": "Core Cellular Machinery",
    "J": "Core Cellular Machinery",
    "L": "Core Cellular Machinery",
    "O": "Core Cellular Machinery",
    "D": "Core Cellular Machinery",
    "Y": "Core Cellular Machinery",
    "Z": "Core Cellular Machinery",
    "B": "Core Cellular Machinery",
    "A": "Core Cellular Machinery",
    "U": "Core Cellular Machinery",
    # Mobilome — interpret with caution
    "X": "Mobilome",
    # Non-characterised — report separately
    "R": "Non-characterised",
    "S": "Non-characterised",
}

print("Imports done")

In [ ]:
all_names = genus_order + list(control_color_map.keys())
html = "<br>".join(
    f"<span style='color:{color_map.get(g, control_color_map.get(g, '#000000'))}; font-weight:600'>{g}</span>"
    for g in all_names
)
display(HTML(html))

## paths and filter toggle


In [ ]:
# ── Filter toggles ─────────────────────────────────────────────────────────────
# True  → use only groups/strains where passes_filter / passes_all_filters == True
# False → use all PV data (original unfiltered behaviour)
SHOW_FILTERED = True

# Retained for compatibility with Figure 3 options; Figure 2 now uses per-figure toggles.
EXCLUDE_UNKNOWNS = True

# ── Paths ──────────────────────────────────────────────────────────────────────
MASTER_GROUPS_PATH  = PROJECT_ROOT / "Results" / "Tables" / "master_groups.csv"
MASTER_STRAINS_PATH = PROJECT_ROOT / "Results" / "Tables" / "master_strains.csv"
BACT_EGGNOG_PATH    = PROJECT_ROOT / "Extraction" / "EGGNOG_ouput" / "output" / "bacteria.emapper.annotations"
BSPECIES_PATH       = PROJECT_ROOT / "Standards" / "SpeciesCallsBacteria.tsv"
# Raw bacteria group summary still needed for locus tag extraction (has _pv_locus columns)
BACT_PHASOMEIT_PATH = PROJECT_ROOT / "Extraction" / "Controls" / "Extraction" / "phasomeit_group_summary.csv"
STRAIN_METADATA     = PROJECT_ROOT / "Extraction" / "lengths and pv per FASTA" / "pv_per_fasta_with_lengths.csv"
OUT_PATH            = PROJECT_ROOT / "Results" / "Figures"

FIG_DIR = Path(OUT_PATH)
FIG_DIR.mkdir(parents=True, exist_ok=True)

SUFFIX = "_filtered" if SHOW_FILTERED else "_unfiltered"
print(f"SHOW_FILTERED    = {SHOW_FILTERED}")
print(f"EXCLUDE_UNKNOWNS = {EXCLUDE_UNKNOWNS}")
print(f"File suffix      : '{SUFFIX}'")
print("Paths set")

## load archaea pv data from master_groups


In [ ]:
master_groups = pd.read_csv(MASTER_GROUPS_PATH, low_memory=False)

arch_phasomeit = master_groups[
    (master_groups["domain"] == "Archaea") &
    master_groups["genus"].isin(genus_order) &
    master_groups["annotated"].eq(True) &
    master_groups["COG_function"].notna()
].copy()

if SHOW_FILTERED:
    arch_phasomeit = arch_phasomeit[arch_phasomeit["passes_filter"] == True]

print(f"master_groups loaded : {len(master_groups):,} rows")
print(f"arch_phasomeit       : {len(arch_phasomeit):,} rows  ({'filtered' if SHOW_FILTERED else 'unfiltered'})")

## load bacteria eggnog annotations


In [ ]:
rows = []
header = None
with open(BACT_EGGNOG_PATH, "r") as f:
    for line in f:
        if line.startswith("##"):
            continue
        elif line.startswith("#query") and header is None:
            header = line.lstrip("#").strip().split("\t")
        elif line.startswith("#"):
            continue
        else:
            rows.append(line.strip().split("\t"))

bact_eggnog = pd.DataFrame(rows, columns=header)
bact_eggnog["sample_id"] = bact_eggnog["query"].str.extract(r'^([^_]+)')
bact_eggnog["locus_tag"] = bact_eggnog["query"].str.replace(r'^[^_]+_', '', regex=True)
bact_eggnog["COG_category_primary"] = bact_eggnog["COG_category"].str.strip().str[0]
bact_eggnog["COG_function"] = bact_eggnog["COG_category_primary"].map(COG_DESCRIPTIONS)

print(f"bact_eggnog: {len(bact_eggnog):,} rows")

In [ ]:
comparator = pd.read_csv(BSPECIES_PATH, sep="\t")
comparator["genus"] = comparator["scientific_name"].str.split().str[0]
comparator = comparator.rename(columns={"sample_accession": "sample_id"})

bact_eggnog = bact_eggnog.merge(comparator[["sample_id", "genus"]], on="sample_id", how="left")
bact_eggnog = bact_eggnog[
    bact_eggnog["genus"].notna() &
    bact_eggnog["COG_function"].notna() &
    ~bact_eggnog["COG_function"].isin(EXCLUDE_COGS)
]

print(f"bact_eggnog after genus mapping: {len(bact_eggnog):,} rows")
print(f"Genera: {bact_eggnog['genus'].unique()}")

## extract bacteria pv loci, one per gene group

instead of melting all strain locus columns (which would give one row per locus
instance per strain and inflate counts for conserved genes), take the first
non-null locus tag per gene group. one row per gene group, matching the
archaea side.

when `SHOW_FILTERED=True`, groups that failed the tract filters are excluded
before linking.


In [ ]:
bact_phasomeit = pd.read_csv(BACT_PHASOMEIT_PATH)
bact_phasomeit["group_num"] = pd.to_numeric(bact_phasomeit["group_num"], errors="coerce")

# Apply filter: keep only groups that passed the tract filter
if SHOW_FILTERED:
    bact_pass = master_groups[
        (master_groups["domain"] == "Bacteria") &
        (master_groups["passes_filter"] == True)
    ][["genus", "group_num"]]
    bact_phasomeit = bact_phasomeit.merge(bact_pass, on=["genus", "group_num"], how="inner")
    print(f"Bacteria groups after filter: {len(bact_phasomeit):,}")

locus_cols = [c for c in bact_phasomeit.columns if c.endswith("_pv_locus")]
print(f"Locus tag columns found: {len(locus_cols)}")

bact_pv_loci = bact_phasomeit[["genus", "group_name"] + locus_cols].copy()
bact_pv_loci["locus_tag"] = bact_pv_loci[locus_cols].apply(
    lambda r: next(
        (v for v in r if pd.notna(v) and v != "" and v != "absent"), None
    ),
    axis=1
)
bact_pv_loci = bact_pv_loci[["genus", "group_name", "locus_tag"]].dropna(subset=["locus_tag"])

print(f"Gene groups with a valid locus tag: {len(bact_pv_loci):,}")
print(f"Genera: {bact_pv_loci['genus'].unique()}")

In [ ]:
bact_eggnog_slim = bact_eggnog[["locus_tag", "COG_function", "COG_category_primary"]].copy()

bact_linked = bact_pv_loci.merge(bact_eggnog_slim, on="locus_tag", how="inner")
bact_linked = bact_linked[
    bact_linked["COG_function"].notna()
]

print(f"bact_linked: {len(bact_linked):,} rows (one per gene group, {'filtered' if SHOW_FILTERED else 'unfiltered'})")
print(f"Genera: {bact_linked['genus'].unique()}")

## load strain counts

used to normalise pv gene group counts by number of strains per genus so
archaea (around 100 strains per genus) and controls (12 strains per genus)
are directly comparable.


In [ ]:
arch_lengths = pd.read_csv(PROJECT_ROOT / "Extraction" / "lengths and pv per FASTA" / "genome_length_counts.csv")
ctrl_lengths = pd.read_csv(PROJECT_ROOT / "Extraction" / "lengths and pv per FASTA" / "genome_length_counts_controls.csv")

all_lengths  = pd.concat([arch_lengths, ctrl_lengths], ignore_index=True)

strain_counts = (
    all_lengths.groupby("genus", as_index=False)
    .agg(n_strains=("fasta_name", "count"))
)

print(strain_counts.to_string(index=False))

## join archaea and bacteria pv-linked data


In [ ]:
arch_slim = arch_phasomeit[["genus", "COG_function", "COG_category_primary"]].copy()
arch_slim["domain"] = "Archaea"

bact_slim = bact_linked[["genus", "COG_function", "COG_category_primary"]].copy()
bact_slim["domain"] = "Bacteria"

for df in [arch_slim, bact_slim]:
    df["broad_role"] = df["COG_category_primary"].map(BROAD_ROLE_MAP)

arch_slim = arch_slim[arch_slim["broad_role"].notna()]
bact_slim = bact_slim[bact_slim["broad_role"].notna()]

all_pv = pd.concat([arch_slim, bact_slim], ignore_index=True)

print(f"Combined PV-linked rows: {len(all_pv):,}")
print(f"Domains: {all_pv['domain'].value_counts().to_dict()}")
all_pv.head(3)


## calculate percent and count_per_strain per function


In [ ]:
def calc_metrics(df, group_col):
    counts = df.groupby(["domain", "genus", group_col]).size().reset_index(name="count")
    totals = df.groupby(["domain", "genus"]).size().reset_index(name="total")
    merged = counts.merge(totals, on=["domain", "genus"])
    merged = merged.merge(strain_counts, on="genus", how="left")
    merged["pct"]              = 100 * merged["count"] / merged["total"]
    merged["count_per_strain"] = merged["count"] / merged["n_strains"]
    return merged

broad_metrics    = calc_metrics(all_pv, "broad_role")
specific_metrics = calc_metrics(all_pv, "COG_function")

print("Broad role metrics (sample):")
print(broad_metrics.head(8).to_string())

## set genus display order


In [ ]:
bact_genera = ["Brucella", "Campylobacter", "Escherichia"]
arch_genera = [g for g in genus_order if g in all_pv["genus"].unique()]
plot_order  = arch_genera + bact_genera
print("Plot order:", plot_order)

## figure 1, pv burden per genome (pv per mb)

with `SHOW_FILTERED=True`, pv counts are recomputed from `master_strains` so
the boxplot reflects the filtered burden. with `False`, values come straight
from `pv_per_fasta_with_lengths.csv`.


In [ ]:
pv_fasta_raw = pd.read_csv(STRAIN_METADATA)

if SHOW_FILTERED:
    # Recompute per-strain PV counts from master_strains filtered rows
    ms = pd.read_csv(MASTER_STRAINS_PATH, low_memory=False)
    pv_rows = ms[
        ms["pv_class"].isin(["intragenic", "upstream"]) &
        (ms["passes_all_filters"] == True)
    ]
    pv_counts = (
        pv_rows.groupby(["domain", "genus", "strain_accession"])["group_num"]
        .nunique().reset_index().rename(columns={"group_num": "PV_count"})
    )
    # Get genome lengths from pv_fasta_raw
    strain_col = next(
        (c for c in pv_fasta_raw.columns if "fasta" in c.lower() or "sample" in c.lower() or "accession" in c.lower()),
        pv_fasta_raw.columns[0]
    )
    len_col = next((c for c in pv_fasta_raw.columns if "length" in c.lower() or "genome" in c.lower()), None)
    genome_lens = pv_fasta_raw[[strain_col, len_col, "genus"]].rename(
        columns={strain_col: "strain_accession", len_col: "genome_length_bp"}
    ).drop_duplicates("strain_accession")
    pv_counts = pv_counts.merge(genome_lens[["strain_accession", "genome_length_bp"]], on="strain_accession", how="left")
    pv_counts["PV_per_Mb"] = pv_counts["PV_count"] / (pv_counts["genome_length_bp"] / 1_000_000)
    # Re-attach genus from genome_lens for strains with 0 PV (they won't appear in pv_rows)
    pv_fasta = pv_fasta_raw[["genus", strain_col]].rename(columns={strain_col: "strain_accession"}).merge(
    pv_counts[["strain_accession", "PV_per_Mb"]], on="strain_accession", how="inner"
    )

    print(f"Figure 1: using filtered PV counts from master_strains")
else:
    pv_fasta = pv_fasta_raw.copy()
    print(f"Figure 1: using pre-computed PV_per_Mb from pv_per_fasta_with_lengths.csv")

pv_fasta = pv_fasta[pv_fasta["genus"].notna() & pv_fasta["PV_per_Mb"].notna()].copy()

fig, ax = plt.subplots(figsize=(12, 4))

for i, genus in enumerate(plot_order):
    data   = pv_fasta[pv_fasta["genus"] == genus]["PV_per_Mb"]
    colour = color_map.get(genus, control_color_map.get(genus, "#999999"))
    ax.boxplot(
        data, positions=[i], widths=0.6, patch_artist=True,
        boxprops=dict(facecolor=colour, alpha=0.7),
        medianprops=dict(color="black", linewidth=1.5),
        whiskerprops=dict(color=colour), capprops=dict(color=colour),
        flierprops=dict(marker="o", markersize=3, color=colour, alpha=0.5)
    )

ax.axvline(len(arch_genera) - 0.5, color="black", linewidth=1.2, linestyle="--", alpha=0.5)
ax.set_xticks(range(len(plot_order)))
ax.set_xticklabels(plot_order, rotation=45, ha="right", fontsize=9)
for tick in ax.get_xticklabels():
    tick.set_color("black")

ax.set_ylabel("PPV loci per Mb", fontsize=14, fontweight="bold")
ax.set_title(
    f"PPV locus density per genus  {'' if SHOW_FILTERED else '[unfiltered]'}",
    fontsize=22, fontweight="bold", color="#1A1A1A"
)
ax.set_xlim(-0.5, len(plot_order) - 0.5)
ax.grid(True, linestyle="--", alpha=0.35)
ax.set_axisbelow(True)

plt.tight_layout()
fig_path = FIG_DIR / f"PV_per_Mb_boxplot{SUFFIX}.png"
plt.savefig(fig_path, dpi=300, bbox_inches="tight")
plt.show()
print(f"Saved: {fig_path}")
#print values as table with median and IQR as well as outliers
summary_stats = []
for genus in plot_order:
    data = pv_fasta[pv_fasta["genus"] == genus]["PV_per_Mb"]
    median = data.median()
    q1 = data.quantile(0.25)
    q3 = data.quantile(0.75)
    iqr = q3 - q1
    outliers = data[(data < q1 - 1.5 * iqr) | (data > q3 + 1.5 * iqr)]
    summary_stats.append({
        "genus": genus,
        "median": median,
        "Q1": q1,
        "Q3": q3,
        "IQR": iqr,
        "outliers": outliers.tolist()
    })
summary_df = pd.DataFrame(summary_stats)
print("\nSummary statistics for PV_per_Mb by genus:")
print(summary_df.to_string(index=False))


## figure 2, broad role (percent left, count_per_strain right)


In [ ]:
# ── BROAD ROLE SETTINGS ───────────────────────────────────────────────────────
BROAD_ROLES_BASE = [
    "Surface & Defense",
    "Env. Response",
    "Adaptive Metabolism",
    "Core Cellular Machinery",
]

BROAD_COLORS = {
    "Surface & Defense":       "#1a6b3a",
    "Env. Response":           "#1f5f99",
    "Adaptive Metabolism":     "#b8620a",
    "Core Cellular Machinery": "#5e3a8c",
    "Non-characterised":       "#8c8c8c",
}

# ── Per-figure toggle — include Non-characterised? ────────────────────────────
SHOW_POORLY_2A = True
SHOW_POORLY_2B = True
SHOW_POORLY_2C = True

# ── Shared figure settings ────────────────────────────────────────────────────
FIG2_BAR_WIDTH       = 0.7   # total width occupied per genus (shared by top stacked + bottom grouped)
FIG2_EDGE_COLOR      = "white"
FIG2_EDGE_WIDTH      = 0.4
FIG2_DIVIDER_X       = len(arch_genera) - 0.5
FIG2_TICK_FONTSIZE   = 11
FIG2_YLABEL_FONTSIZE = 13
FIG2_TITLE_FONTSIZE  = 20
FIG2_LEGEND_FONTSIZE = 12
FIG2_SAVE_DPI        = 300
FIG2_GRID_LINEWIDTH = 0.9 
FIG2_GRID_COLOR = "#555555" 
FIG2_GRID_ALPHA = 0.55 # 

# ── Helper: build roles list and pivot ────────────────────────────────────────
def _build_pivot(metric_col, show_poorly):
    roles = BROAD_ROLES_BASE + (["Non-characterised"] if show_poorly else [])
    pivot = (
        broad_metrics
        .pivot_table(index="genus", columns="broad_role", values=metric_col, aggfunc="mean")
        .reindex(index=plot_order, columns=roles)
        .fillna(0)
    )
    return roles, pivot

# ── Draw stacked bars (top panel) ─────────────────────────────────────────────
def _draw_stacked(ax, pivot, roles):
    bottom = np.zeros(len(plot_order))
    for role in roles:
        vals = pivot[role].values
        ax.bar(range(len(plot_order)), vals, bottom=bottom,
               color=BROAD_COLORS[role], edgecolor=FIG2_EDGE_COLOR,
               linewidth=FIG2_EDGE_WIDTH, width=FIG2_BAR_WIDTH)
        bottom += vals

# ── Draw grouped thin bars (bottom panel) ─────────────────────────────────────
# Each role gets an equal-width slice of FIG2_BAR_WIDTH, bars touch within genus,
# gap between genera is the remaining space (1.0 - FIG2_BAR_WIDTH).
def _draw_grouped(ax, pivot, roles):
    n = len(roles)
    thin_w = FIG2_BAR_WIDTH / n
    for j, role in enumerate(roles):
        # Centre each thin bar within its slot
        x = np.arange(len(plot_order), dtype=float) \
            - FIG2_BAR_WIDTH / 2 \
            + (j + 0.5) * thin_w
        vals = pivot[role].values
        # Zero values would break log scale — replace with NaN so bar is simply absent
        vals_plot = np.where(vals > 0, vals, np.nan)
        ax.bar(x, vals_plot, width=thin_w,
               color=BROAD_COLORS[role], edgecolor="none",
               linewidth=0)

# ── Shared axis styling ───────────────────────────────────────────────────────
def _style_ax(ax, ylabel, yscale="linear", show_xticklabels=False):
    ax.axvline(FIG2_DIVIDER_X, color="black", linewidth=1.2, linestyle="--", alpha=0.6)
    ax.set_xticks(range(len(plot_order)))
    if show_xticklabels:
        ax.set_xticklabels(plot_order, rotation=45, ha="right", fontsize=FIG2_TICK_FONTSIZE)
        for tick in ax.get_xticklabels():
            tick.set_color("black")
    else:
        ax.set_xticklabels([])
    ax.set_ylabel(ylabel, fontsize=FIG2_YLABEL_FONTSIZE, fontweight="bold")
    ax.set_xlim(-0.5, len(plot_order) - 0.5)
    if yscale == "log":
        ax.set_yscale("log")
    ax.grid(True, linestyle="--", alpha=FIG2_GRID_ALPHA,
    linewidth=FIG2_GRID_LINEWIDTH, color=FIG2_GRID_COLOR, axis="y")
    ax.set_axisbelow(True)

# ─────────────────────────────────────────────────────────────────────────────
# Figures 2A + 2B — combined panel
# ─────────────────────────────────────────────────────────────────────────────
roles_2a, pct_pivot = _build_pivot("pct",              SHOW_POORLY_2A)
roles_2b, cps_pivot = _build_pivot("count_per_strain", SHOW_POORLY_2B)

fig2, (ax2a, ax2b) = plt.subplots(
    2, 1, figsize=(14, 9), sharex=True,
    gridspec_kw={"height_ratios": [2, 1], "hspace": 0.06}
)

# ── Top panel — stacked % ─────────────────────────────────────────────────────
_draw_stacked(ax2a, pct_pivot, roles_2a)
ax2a.set_ylim(0, 100)
_style_ax(ax2a,
          ylabel="% of total PPV-linked\ngene groups",
          show_xticklabels=False)

# ── Bottom panel — grouped thin bars, one per category per genus ──────────────
_draw_grouped(ax2b, cps_pivot, roles_2b)
_style_ax(ax2b,
          ylabel="Mean number of PPV\ngene groups per strain",
          yscale="log",
          show_xticklabels=True)

# ── Single figure title ───────────────────────────────────────────────────────
fig2.suptitle(
    "Functional composition and strain-normalised\nabundance of PPV gene groups across genera",
    fontsize=FIG2_TITLE_FONTSIZE, fontweight="bold", color="#1A1A1A", y=1.01
)

# ── Legend between title and top panel ───────────────────────────────────────
legend_roles = roles_2a
handles = [mpatches.Patch(color=BROAD_COLORS[r], label=r) for r in legend_roles]
fig2.legend(
    handles=handles,
    loc="lower center",
    bbox_to_anchor=(0.507, 0.89),
    ncol=len(legend_roles),
    fontsize=FIG2_LEGEND_FONTSIZE,
    frameon=True,
    framealpha=0.9,
    edgecolor="#cccccc",
)

plt.savefig(
    FIG_DIR / f"COG_2AB_combined{SUFFIX}.png",
    dpi=FIG2_SAVE_DPI, bbox_inches="tight"
)
plt.show()

# ─────────────────────────────────────────────────────────────────────────────
# Figure 2C — reference: COG function count per broad role
# ─────────────────────────────────────────────────────────────────────────────
roles_2c = BROAD_ROLES_BASE + (["Non-characterised"] if SHOW_POORLY_2C else [])
func_counts_2c = {
    role: sum(1 for r in BROAD_ROLE_MAP.values() if r == role )
    for role in roles_2c
}

fig2c, ax2c = plt.subplots(figsize=(5, 4.5))
bottom = 0
for role in roles_2c:
    count = func_counts_2c[role]
    ax2c.bar(0, count, bottom=bottom, color=BROAD_COLORS[role],
             edgecolor="white", linewidth=0.8, width=0.6)
    ax2c.text(0, bottom + count / 2, str(count),
              ha="center", va="center", fontsize=13, fontweight="bold", color="white")
    bottom += count

handles = [mpatches.Patch(color=BROAD_COLORS[r], label=r) for r in roles_2c]
ax2c.legend(handles=handles, loc="upper left", bbox_to_anchor=(1.05, 1),
            fontsize=9, frameon=False)
ax2c.axis("off")
plt.tight_layout()
plt.savefig(FIG_DIR / "COG_2C_broad_role_composition.png", dpi=FIG2_SAVE_DPI, bbox_inches="tight")
plt.show()
#print the table of per genus values for both plots, % for top and count_per_strain for bottom
print("Broad role metrics per genus:")
broad_metrics_print = broad_metrics.copy()
broad_metrics_print["pct"] = broad_metrics_print["pct"].round(2)
broad_metrics_print["count_per_strain"] = broad_metrics_print["count_per_strain"].round(3)
print(broad_metrics_print[["domain", "genus", "broad_role", "pct", "count_per_strain"]].to_string(index=False))

## figure 3, specific-function heatmaps and compact key

- 3a: percent of pv repertoire per cog function
- 3b: pv gene groups per strain per cog function
- 3c: cog letter key


In [ ]:
# ── USER SETTINGS ─────────────────────────────────────────────────────────────
FONT_FAMILY = "Arial"

# Order functions grouped by the new broad roles (matches Figure 2)
COG_FUNCTION_ORDER = [
    # Surface & Defense
    "Cell wall/membrane/envelope biogenesis",
    "Cell motility",
    "Extracellular structures",
    "Defense mechanisms",
    # Environmental Response
    "Signal transduction",
    "Transcription",
    # Adaptive Metabolism
    "Inorganic ion transport and metabolism",
    "Carbohydrate transport and metabolism",
    "Secondary metabolite biosynthesis and catabolism",
    "Amino acid transport and metabolism",
    # Core Cellular Machinery
    "Energy production and conversion",
    "Coenzyme transport and metabolism",
    "Lipid transport and metabolism",
    "Nucleotide transport and metabolism",
    "Translation",
    "Replication, recombination and repair",
    "Post-translational modification and chaperones",
    "Cell cycle control and division",
    "Intracellular trafficking and secretion",
    # Mobilome
    "Mobilome: prophages and transposons",
]

BROAD_GROUP_ORDER = ["Surface & Defense", "Env. Response", "Adaptive Metabolism",
                     "Core Cellular Machinery", "Mobilome"]
if not EXCLUDE_UNKNOWNS:
    BROAD_GROUP_ORDER = BROAD_GROUP_ORDER + ["Non-characterised"]

GENUS_FONTSIZE = 11;  GENUS_ROTATION = 0;  GENUS_FONTWEIGHT = "normal";  GENUS_LABEL_COLOR = "#1A1A1A"
XLETTER_FONTSIZE = 14;  XLETTER_FONTWEIGHT = "bold";  XAXIS_LABEL = "COG category (letter code)"
YAXIS_LABEL_TEXT = "";  YAXIS_LABEL_FONTSIZE = 10;  YAXIS_LABEL_FONTWEIGHT = "bold"
YAXIS_LABELPAD = 8;  YAXIS_LABEL_X = -0.10;  YAXIS_LABEL_Y = 0.50

FIG_WIDTH_MIN = 12;  FIG_WIDTH_PER_FUNC = 0.75;  FIG_HEIGHT_PER_GENUS = 0.2;  FIG_HEIGHT_PADDING = 2

ARCH_BACT_DIVIDER_COLOR = "black";  ARCH_BACT_DIVIDER_LINEWIDTH = 1.5;  ARCH_BACT_DIVIDER_LINESTYLE = "--"
GROUP_DIVIDER_COLOR = "#444444";  GROUP_DIVIDER_LINEWIDTH = 1.2;  GROUP_DIVIDER_LINESTYLE = "-";  GROUP_DIVIDER_ALPHA = 0.5
GROUP_LABEL_FONTSIZE = 9;  GROUP_LABEL_COLOR = "black";  GROUP_LABEL_FONTWEIGHT = "normal" #change to italic non bold by = "i"
GROUP_LABEL_FONTSTYLE = "italic";  GROUP_LABEL_Y = 1

TITLE_A = "COG functional bias of PPV loci by genus"
TITLE_B = "Abundance of PPV gene groups by COG function per genome assembly"
TITLE_C = "COG category letter key"
TITLE_FONTSIZE = 20;  TITLE_FONTWEIGHT = "bold";  TITLE_COLOR = "#1A1A1A";  TITLE_PAD = 28
CBAR_A_LABEL = "% of genus PPV total";  CBAR_B_LABEL = "PPV gene groups per strain";  CBAR_LABEL_FONTSIZE = 13
CBAR_SHRINK = 0.5;  CBAR_PAD = 0.01

PCT_CMAP = "YlOrRd";  PCT_VMIN = 0
CPS_UPPER_FLOOR = 60.0
CPS_COLOUR_BREAKS = [0, 0.5, 3, 20]
CPS_COLOUR_STOPS  = ["#ffffff", "#2166ac", "#fdd835", "#d9966a", "#981a1a"]

KEY_COLUMNS = 2;  KEY_FIG_WIDTH = 8;  KEY_ROW_HEIGHT = 0.28;  KEY_HEIGHT_PADDING = 1.0
KEY_FONTSIZE = 11;  KEY_TITLE_FONTSIZE = 13;  KEY_LEFT_MARGIN = 0.04;  KEY_COL_GAP = 0.50
KEY_LINE_SPACING = 1.00;  KEY_TEXT_COLOR = "#111111"
KEY_HEADER_FONTWEIGHT = "bold";  KEY_ENTRY_FONTWEIGHT = "normal"

FIG_NAME_A = f"COG_3A_pct_heatmap{SUFFIX}.png"
FIG_NAME_B = f"COG_3B_cps_heatmap{SUFFIX}.png"
FIG_NAME_C = f"COG_3C_letter_key{SUFFIX}.png"
SAVE_DPI = 300;  SAVE_BBOX = "tight"

# ── DATA PREP ─────────────────────────────────────────────────────────────────
# Map each function name → its broad role using BROAD_ROLE_MAP (same as Figure 2)
DESC_TO_LETTER   = {v: k for k, v in COG_DESCRIPTIONS.items()}
LETTER_TO_BROAD  = BROAD_ROLE_MAP   # letter → broad role
COG_BROAD_MEMBERSHIP = {
    f: LETTER_TO_BROAD.get(DESC_TO_LETTER.get(f), None)
    for f in COG_FUNCTION_ORDER
}

funcs_available = [
    f for f in COG_FUNCTION_ORDER
    if f in specific_metrics["COG_function"].unique()
    and (EXCLUDE_UNKNOWNS is False or COG_BROAD_MEMBERSHIP.get(f) not in ["Non-characterised", None])
]
x_letters       = [DESC_TO_LETTER.get(f, f) for f in funcs_available]
spec_pct_pivot  = specific_metrics.pivot_table(index="genus", columns="COG_function", values="pct",
                    aggfunc="mean").reindex(index=plot_order)[funcs_available].fillna(0)
spec_cps_pivot  = specific_metrics.pivot_table(index="genus", columns="COG_function", values="count_per_strain",
                    aggfunc="mean").reindex(index=plot_order)[funcs_available].fillna(0)
n_funcs  = len(funcs_available)
n_genera = len(plot_order)

# Colour for each group divider label — reuse BROAD_COLORS from Figure 2
BROAD_COLORS_3 = {**BROAD_COLORS, "Non-characterised": "#720909"}

# ── HELPERS ───────────────────────────────────────────────────────────────────
def heatmap_figsize(nf, ng):
    return (max(FIG_WIDTH_MIN, nf * FIG_WIDTH_PER_FUNC), ng * FIG_HEIGHT_PER_GENUS + FIG_HEIGHT_PADDING)

def apply_genus_labels(ax, names):
    ax.set_yticks(range(len(names)))
    ax.set_yticklabels(names, fontsize=GENUS_FONTSIZE, fontweight=GENUS_FONTWEIGHT)
    for tick, g in zip(ax.get_yticklabels(), names):
        tick.set_color(color_map.get(g, control_color_map.get(g, "#000000")) if GENUS_LABEL_COLOR == "colormap" else "#141414")
        tick.set_rotation(GENUS_ROTATION)

def add_broad_dividers_and_labels(ax):
    cursor = 0
    for broad in BROAD_GROUP_ORDER:
        group_funcs = [f for f in funcs_available if COG_BROAD_MEMBERSHIP.get(f) == broad]
        if not group_funcs: continue
        n = len(group_funcs)
        if cursor > 0:
            ax.axvline(cursor - 0.5, color=GROUP_DIVIDER_COLOR, linewidth=GROUP_DIVIDER_LINEWIDTH,
                       linestyle=GROUP_DIVIDER_LINESTYLE, alpha=GROUP_DIVIDER_ALPHA)
        ax.text(cursor + n / 2 - 0.5, GROUP_LABEL_Y, broad, ha="center", va="bottom",
                fontsize=GROUP_LABEL_FONTSIZE, color=GROUP_LABEL_COLOR,
                fontweight=GROUP_LABEL_FONTWEIGHT, style=GROUP_LABEL_FONTSTYLE,
                transform=ax.get_xaxis_transform())
        cursor += n

def apply_common_heatmap_format(ax):
    ax.axhline(len(arch_genera) - 0.5, color=ARCH_BACT_DIVIDER_COLOR,
               linewidth=ARCH_BACT_DIVIDER_LINEWIDTH, linestyle=ARCH_BACT_DIVIDER_LINESTYLE)
    apply_genus_labels(ax, plot_order)
    ax.set_xticks(range(n_funcs))
    ax.set_xticklabels(x_letters, fontsize=XLETTER_FONTSIZE, fontweight=XLETTER_FONTWEIGHT)
    ax.set_xlabel(XAXIS_LABEL, fontsize=11)
    ax.set_ylabel(YAXIS_LABEL_TEXT, fontsize=YAXIS_LABEL_FONTSIZE, fontweight=YAXIS_LABEL_FONTWEIGHT, labelpad=YAXIS_LABELPAD)
    ax.yaxis.set_label_coords(YAXIS_LABEL_X, YAXIS_LABEL_Y)
    add_broad_dividers_and_labels(ax)

# ── CPS COLOR NORM ────────────────────────────────────────────────────────────
cps_upper  = max(CPS_UPPER_FLOOR, float(np.nanmax(spec_cps_pivot.values)))
cps_breaks = CPS_COLOUR_BREAKS + [cps_upper]
cps_scale_positions = np.linspace(0, 1, len(cps_breaks))
cps_cmap = mcolors.LinearSegmentedColormap.from_list("pv_cps_sensitive",
    list(zip(cps_scale_positions, CPS_COLOUR_STOPS)), N=512)
cps_norm = mcolors.FuncNorm(
    (lambda v: np.interp(v, cps_breaks, cps_scale_positions),
     lambda v: np.interp(v, cps_scale_positions, cps_breaks)),
    vmin=cps_breaks[0], vmax=cps_breaks[-1])

# ── FIGURE 3A ─────────────────────────────────────────────────────────────────
fig_a, ax_a = plt.subplots(figsize=heatmap_figsize(n_funcs, n_genera))
im_a = ax_a.imshow(spec_pct_pivot.values, cmap=PCT_CMAP, aspect="auto",
                   vmin=PCT_VMIN, vmax=spec_pct_pivot.values.max())
apply_common_heatmap_format(ax_a)
cbar_a = plt.colorbar(im_a, ax=ax_a, shrink=CBAR_SHRINK, pad=CBAR_PAD)
cbar_a.set_label(CBAR_A_LABEL, fontsize=CBAR_LABEL_FONTSIZE)
ax_a.set_title(TITLE_A, fontsize=TITLE_FONTSIZE, fontweight=TITLE_FONTWEIGHT, color=TITLE_COLOR, pad=TITLE_PAD)
plt.tight_layout()
plt.savefig(FIG_DIR / FIG_NAME_A, dpi=SAVE_DPI, bbox_inches=SAVE_BBOX)
plt.show()
print(f"Saved: {FIG_DIR / FIG_NAME_A}")

# ── FIGURE 3B ─────────────────────────────────────────────────────────────────
fig_b, ax_b = plt.subplots(figsize=heatmap_figsize(n_funcs, n_genera))
im_b = ax_b.imshow(spec_cps_pivot.values, cmap=cps_cmap, aspect="auto", norm=cps_norm)
apply_common_heatmap_format(ax_b)
cbar_b = plt.colorbar(im_b, ax=ax_b, shrink=CBAR_SHRINK, pad=CBAR_PAD)
cbar_b.set_ticks(cps_breaks[:-1])
cbar_b.set_label(CBAR_B_LABEL, fontsize=CBAR_LABEL_FONTSIZE)
ax_b.set_title(TITLE_B, fontsize=TITLE_FONTSIZE, fontweight=TITLE_FONTWEIGHT, color=TITLE_COLOR, pad=TITLE_PAD)
plt.tight_layout()
plt.savefig(FIG_DIR / FIG_NAME_B, dpi=SAVE_DPI, bbox_inches=SAVE_BBOX)
plt.show()
print(f"Saved: {FIG_DIR / FIG_NAME_B}")

# ── FIGURE 3C — COG letter key, 3-column newspaper layout ────────────────────
import textwrap

WRAP_CHARS = 58     # wider columns on 14-inch figure need less wrapping
LINE_H_IN  = 0.20   # physical height per text row in inches
SPACER_H   = 0.5    # spacer between categories = fraction of a row
COL_PAD    = 0.05   # left padding within each column

# Build flat list with pre-wrapped lines
all_lines = []   # (text, is_header)
#for broad in active_broads:
#    all_lines.append((broad, True))
#    for letter, func in key_by_broad[broad]:
##        for line in textwrap.wrap(f"{letter} \u2013 {func}", width=WRAP_CHARS):
#            all_lines.append((line, False))
#    all_lines.append(("", False))   # spacer

#while all_lines and all_lines[-1][0] == "":
#    all_lines.pop()

# Split into 3 columns at spacer boundaries nearest each third
# Backward search so no category gets split from its header
n_total  = len(all_lines)
col_data = []
prev     = 0
for target in [n_total // 3, 2 * n_total // 3]:
    split_at = target
    for i in range(min(target, n_total - 1), max(prev, 0), -1):
        if all_lines[i][0] == "":
            split_at = i + 1
            break
    col_data.append(all_lines[prev:split_at])
    prev = split_at
col_data.append(all_lines[prev:])

# Compute true height of each column in row-units (spacers = SPACER_H, text = 1.0)
def col_height(lines):
    return sum(SPACER_H if not t else 1.0 for t, _ in lines)

max_y     = max(col_height(c) for c in col_data)
fig_key_w = 14
fig_key_h = 0.45 + max_y * LINE_H_IN + 0.2   # top pad + content + bottom pad

fig_c, ax_c = plt.subplots(figsize=(fig_key_w, fig_key_h))
ax_c.set_xlim(0, 3)
ax_c.set_ylim(max_y, 0)   # y=0 at top, increases downward
ax_c.axis("off")

# Subtle column separators
for i in range(1, 3):
    ax_c.axvline(i, color="#e8e8e8", linewidth=0.8, zorder=0)

for col_idx, lines in enumerate(col_data):
    x = col_idx + COL_PAD
    y = 0.5   # start half a row down from the top
    for text, is_header in lines:
        if not text:
            y += SPACER_H
            continue
        ax_c.text(
            x, y,
            text,
            fontsize   = 9,
            fontweight = "bold" if is_header else "normal",
            color      = "#111111",
            va="center", ha="left",
            fontfamily = "Arial",
        )
        y += 1.0

ax_c.set_title(
    TITLE_C,
    fontsize=KEY_TITLE_FONTSIZE, fontweight=KEY_HEADER_FONTWEIGHT,
    pad=6, color="#1A1A1A"
)

plt.tight_layout()
plt.savefig(FIG_DIR / FIG_NAME_C, dpi=SAVE_DPI, bbox_inches=SAVE_BBOX)
plt.show()


#view table of cells  percentages and counts for both heatmaps for all genera and functions, with columns renamed to indicate which metric they represent
combined_table = spec_pct_pivot.copy()
combined_table.columns = [f"{col}_pct" for col in combined_table.columns]
for col in spec_cps_pivot.columns:
    combined_table[f"{col}_cps"] = spec_cps_pivot[col]
combined_table = combined_table.reset_index()
print("Combined table of % and count_per_strain for all genera and functions:")
print(combined_table.to_string(index=False))


## save tables


In [ ]:
table_dir = FIG_DIR.parent / "Tables"
table_dir.mkdir(parents=True, exist_ok=True)

broad_metrics.to_csv(table_dir / f"COG_broad_role_metrics{SUFFIX}.csv", index=False)
specific_metrics.to_csv(table_dir / f"COG_specific_function_metrics{SUFFIX}.csv", index=False)

print(f"Saved: {table_dir / f'COG_broad_role_metrics{SUFFIX}.csv'}")
print(f"Saved: {table_dir / f'COG_specific_function_metrics{SUFFIX}.csv'}")

In [ ]:
# ── SETTINGS — edit here ─────────────────────────────────────────────────────

# How many top functions to show per genus
TOP_N = 10

# Metric plotted on y-axis:
#   "group_count"   → number of distinct PV gene groups with this function
#   "total_pv_sum"  → sum of total_pv (total SSR-PV sites) across all groups
METRIC = "group_count"

# Function labels to exclude (case-insensitive exact match)
# Add strings here to suppress uninformative entries
EXCLUDE_LABELS = {
    "hypothetical protein",
    "no annotation data",
    "No data, identified by script",
}

# Truncate long function labels to this many characters (avoids x-axis overflow)
LABEL_MAX_LEN = 45

# Use each genus's own colour from color_map / control_color_map
# Set False to use the flat fallback colours below
USE_GENUS_COLOR = True
FALLBACK_COLOR_ARCH = "#1a6b3a"
FALLBACK_COLOR_BACT = "#2d2d2d"

# Figure dimensions per genus plot
FIG_WIDTH  = 11
FIG_HEIGHT = 5

# Axis
LOG_SCALE         = True
LABEL_ROTATION    = 45
BAR_EDGE_COLOR    = "white"
BAR_EDGE_WIDTH    = 0.4
TICK_FONTSIZE     = 9
YLABEL_FONTSIZE   = 10
TITLE_FONTSIZE    = 12

# Save
FIG4_SAVE_DPI = 300
FIG4_SUBDIR   = "Figure4_top_functions"   # subfolder inside FIG_DIR


# individual top-10 function plots from prokka annotations


In [ ]:
# -- DATA: load, filter, wrangle ------------------------------------------------
import re

def _norm_label(s):
    # Collapse whitespace and case-normalize so exclusions are robust to casing/spacing.
    return re.sub(r"\s+", " ", str(s)).strip().casefold()

# Depends on: master_groups (loaded earlier), plot_order, arch_genera,
#             SHOW_FILTERED, color_map, control_color_map

# -- Subset master_groups to relevant genera ------------------------------------
func_raw = master_groups[
    master_groups["genus"].isin(plot_order) &
    master_groups["likely_function"].notna()
] .copy()

if SHOW_FILTERED:
    func_raw = func_raw[func_raw["passes_filter"] == True]

# -- Remove uninformative labels -------------------------------------------------
func_raw["likely_function"] = func_raw["likely_function"].astype(str).str.strip()
exclude_labels_norm = {_norm_label(x) for x in EXCLUDE_LABELS}
func_raw = func_raw[
    ~func_raw["likely_function"].apply(_norm_label).isin(exclude_labels_norm)
]

# -- Compute metric per genus x function ----------------------------------------
if METRIC == "group_count":
    func_metrics = (
        func_raw
        .groupby(["genus", "likely_function"])
.size()
        .reset_index(name="value")
    )
    y_label = "Putative PV gene group count"
else:
    func_metrics = (
        func_raw
        .groupby(["genus", "likely_function"])["total_pv"]
.sum()
        .reset_index(name="value")
    )
    y_label = "Sum of total PV sites (sum of total_pv)"

if LOG_SCALE:
    y_label += " - log scale"

# -- Per-genus top N, sorted descending -----------------------------------------
top_funcs = (
    func_metrics
.sort_values("value", ascending=False)
    .groupby("genus", group_keys=False)
.head(TOP_N)
)

print(f"METRIC         : {METRIC}")
print(f"SHOW_FILTERED  : {SHOW_FILTERED}")
print(f"Genera covered : {sorted(top_funcs['genus'].unique())}")
print(f"Rows in top_funcs: {len(top_funcs):,}")

# Quick sanity check for the requested exclusion
excluded_hits = top_funcs[top_funcs["likely_function"].apply(_norm_label) == _norm_label("No data, identified by script")]
print(f"Excluded label present in top_funcs?: {len(excluded_hits) > 0}")

In [ ]:
import matplotlib.pyplot as plt
from matplotlib import rcParams

# ─────────────────────────────────────────────────────────────────────────────
# FIGURE SPEC
#   Target           : Word supplementary figure
#   Figure width     : 7.8 in (requires <1" page margins or landscape supp)
#   Body font (doc)  : Arial 11 pt — in-figure text 10–13 pt within that range
#   Output           : PNG @ 400 dpi; pdf.fonttype=42 keeps text editable for
#                      later vector export
#   Layout           : 5 horizontal-bar panels (one per genus), stacked
# ─────────────────────────────────────────────────────────────────────────────

rcParams.update({
    "font.family":     "Arial",
    "font.size":        11,
    "axes.titlesize":   13,
    "axes.labelsize":   12,
    "xtick.labelsize":  11,
    "ytick.labelsize":  12,
    "axes.linewidth":  0.6,
    "pdf.fonttype":    42,
})

fig4_dir = FIG_DIR / FIG4_SUBDIR
fig4_dir.mkdir(parents=True, exist_ok=True)

# ── Genera to plot (order = display order, top → bottom) ────────────────────
SUPP_GENERA = [
    "Methanobrevibacter_A",
    "Methanococcus",
    "Methanohalophilus",
    "Nitrosopelagicus",
    "Methanobacterium_D",
]

genera_to_plot = [g for g in SUPP_GENERA
                  if not top_funcs[top_funcs["genus"] == g].empty]

missing = [g for g in SUPP_GENERA if g not in genera_to_plot]
if missing:
    print(f"  ⚠ no data for: {', '.join(missing)} — skipped")

# ── Layout constants ────────────────────────────────────────────────────────
FIG_WIDTH_IN     = 7.8
PANEL_HEIGHT     = 2.5
EXTRA_BOTTOM_IN  = 0.4    # reserved strip below last panel for supxlabel
BAR_PAD_FRAC     = 1.25   # x-limit padding → ~10% narrower bar area
TITLE_PAD        = 0.006  # fig-coord gap between panel top and centred title

fig_height = PANEL_HEIGHT * len(genera_to_plot) + EXTRA_BOTTOM_IN

fig, axes = plt.subplots(
    nrows=len(genera_to_plot), ncols=1,
    figsize=(FIG_WIDTH_IN, fig_height),
    layout="constrained",
)
if len(genera_to_plot) == 1:
    axes = [axes]

# Generous breathing room: between panels (room for centred titles) and
# around the axes (room between supxlabel and the x-tick numbers above it)
fig.get_layout_engine().set(h_pad=0.10, w_pad=0.05, hspace=0.08)

for ax, genus in zip(axes, genera_to_plot):
    gdata = (top_funcs[top_funcs["genus"] == genus]
             .sort_values("value", ascending=True)
             .reset_index(drop=True))

    bar_color = (color_map.get(genus, control_color_map.get(genus, "#444444"))
                 if USE_GENUS_COLOR
                 else (FALLBACK_COLOR_ARCH if genus in arch_genera
                       else FALLBACK_COLOR_BACT))

    labels = gdata["likely_function"].tolist()
    values = gdata["value"].tolist()
    y      = range(len(labels))

    bars = ax.barh(y, values,
                   color=bar_color,
                   edgecolor=BAR_EDGE_COLOR,
                   linewidth=0.4)

    ax.set_yticks(list(y))
    ax.set_yticklabels(labels)

    # value labels at bar ends
    xmax = max(values)
    ax.set_xlim(0, xmax * BAR_PAD_FRAC)
    for bar, v in zip(bars, values):
        ax.text(v + xmax * 0.015,
                bar.get_y() + bar.get_height() / 2,
                f"{v}", va="center", ha="left", fontsize=10)

    # spines / ticks / grid
    for side in ("top", "right"):
        ax.spines[side].set_visible(False)
    ax.tick_params(axis="y", length=0)
    ax.tick_params(axis="x", length=3)
    ax.set_axisbelow(True)
    ax.grid(axis="x", linewidth=0.4, alpha=0.4)

# ── Figure-level axis labels (regular weight: journal convention) ───────────


# ── Centre genus titles on the FIGURE (not on the bar area) ─────────────────
fig.canvas.draw()   # resolve constrained_layout so axis positions are final
for ax, genus in zip(axes, genera_to_plot):
    pos = ax.get_position()
    fig.text(0.5, pos.y1 + TITLE_PAD, genus,
             ha="center", va="bottom",
             fontweight="bold", style="italic", fontsize=13)

save_path = fig4_dir / f"top_functions_supp_archaea{SUFFIX}.png"
fig.savefig(save_path, dpi=400)
plt.show()
print(f"Saved: {save_path}")

In [ ]:
# Print Top-10 function rankings (value + rank) for selected genera
selected_genera_input = [
    "methanobacterium_D",
    "Methanococcus",
    "Methanobrevibacter_A",
    "Methanohalophilus",
    "Nitrosopelagicus",
]

# Resolve user-provided names against available genus labels (case-insensitive)
available_genera = sorted(top_funcs["genus"].dropna().unique())
lookup = {g.casefold(): g for g in available_genera}
selected_genera = [lookup.get(g.casefold(), g) for g in selected_genera_input]

print("Top-10 function rankings for selected genera")
print(f"Metric: {METRIC}")
print(f"Rows in top_funcs: {len(top_funcs):,}")

for genus in selected_genera:
    gdata = (
        top_funcs[top_funcs["genus"] == genus]
        .sort_values("value", ascending=False)
        .head(10)
        .copy()
    )

    print("\n" + "=" * 90)
    print(f"{genus}")

    if gdata.empty:
        print("No rows found for this genus in top_funcs.")
        continue

    gdata["rank"] = gdata["value"].rank(method="first", ascending=False).astype(int)
    gdata = gdata[["rank", "likely_function", "value"]].sort_values("rank")

    print(gdata.to_string(index=False))